# Step 2 — Ring-geometry inference with **emcee** (MCMC)

Same model, likelihood and priors as the dynesty notebook, sampled with `emcee` ensemble MCMC
instead. Use this as an independent cross-check of the posterior; it does **not** produce the
Bayesian evidence (use the dynesty notebook for $\ln\mathcal{Z}$).

## 0. Environment

In [ ]:
# ── Bootstrap: make the sibling packages importable without installation ────
# The pipeline uses three packages that live in the repository, uninstalled:
#   exorings, geotrans   (repo root)      photoring   (pipeline/)
# We locate the repo root and pipeline/ robustly from the current working dir
# (Jupyter / nbconvert / papermill all run notebooks from pipeline/).
import sys, pathlib
_HERE   = pathlib.Path.cwd().resolve()
_cands  = [_HERE, *_HERE.parents]
_NB_DIR = next((c for c in _cands if (c / "photoring").is_dir()), _HERE)
_REPO   = next((c for c in _cands if (c / "exorings").is_dir()), _NB_DIR.parent)
for _p in (str(_REPO), str(_NB_DIR)):
    if _p not in sys.path:
        sys.path.insert(0, _p)
print("repo root :", _REPO)
print("pipeline  :", _NB_DIR)

In [ ]:
import numpy as np
import warnings, os, time
warnings.filterwarnings("ignore")
# Limit BLAS threads before heavy numerics (pool workers each add threads).
for _v in ["OMP_NUM_THREADS","MKL_NUM_THREADS","OPENBLAS_NUM_THREADS",
           "VECLIB_MAXIMUM_THREADS","NUMEXPR_NUM_THREADS"]:
    os.environ.setdefault(_v, "2")

import photoring as pr
import photoring.plotting as plot
plot.apply_style()

import emcee
print("emcee", emcee.__version__)

## 1. USER CONFIGURATION  ← edit here (papermill-injected)

In [ ]:
CASE   = "kepler_51"
PLANET = "d"

PLANET_PARAMS = {
    "d": dict(B_FIXED=0.0030, B_SIGMA=2 * 0.0950, p_mean_ref=0.09857, p_prior_lo=0.23),
    "b": dict(B_FIXED=0.0740, B_SIGMA=2 * 0.0720, p_mean_ref=0.07225, p_prior_lo=0.33),
}

KDE_CONFIG = {"observables": ["delta", "rho_obs", "T14"], "N_KDE": 5000, "seed_kde": 123}

MCMC_CONFIG = {"nwalkers": 64, "nsteps": 10000, "burnin": 2000, "thin": 50,
               "seed": 2026, "use_pool": True, "n_procs": 3}

MODEL_CONFIG = {
    "B_FREE": True, "B_FIXED": PLANET_PARAMS[PLANET]["B_FIXED"],
    "B_SIGMA": PLANET_PARAMS[PLANET]["B_SIGMA"],
    "RHO_TRUE_FREE": True, "RHO_TRUE_FIXED": None,
    "FI_FIXED": 1.0, "FE_MAX": 10.0,
    "TAU_FREE": False, "TAU_FIXED": 1.0, "TAU_PRIOR_LO": 0.1, "TAU_PRIOR_HI": 10.0,
    "P_FREE": True, "p_mean_ref": PLANET_PARAMS[PLANET]["p_mean_ref"],
    "p_prior_lo": PLANET_PARAMS[PLANET]["p_prior_lo"], "p_prior_hi": 1.0,
    "FORWARD_MODEL": "exorings",
}

## 2. Build the model

In [ ]:
paths = pr.CasePaths(CASE)
FORWARD_MODEL = str(MODEL_CONFIG.get("FORWARD_MODEL", "exorings")).lower()
paths.ensure_outputs(FORWARD_MODEL)

# Load the case's derived observables, rho_true samples and inverse-CDF grid.
data  = pr.load_case_data(paths, PLANET)
model = pr.PhotoRingModel(
    data["ttv"], data["rho_true_gcc_samples"], MODEL_CONFIG, KDE_CONFIG,
    rho_grid=data["rho_grid"], rho_cdf=data["rho_cdf"], p_fixed=data["P_fixed"],
)
print(f"Planet {PLANET}: {len(data['ttv']['delta'])} TTV samples | P_fixed={data['P_fixed']:.6f} d")
print(f"NDIM={model.NDIM}  params={model.PARAM_NAMES}")

In [ ]:
_kt = "-".join(model.observables)
RUN_TAG = (f"{CASE}_{PLANET}_MCMC_{FORWARD_MODEL}_kde_{_kt}"
           f"_nw{MCMC_CONFIG['nwalkers']}_ns{MCMC_CONFIG['nsteps']}"
           f"_bi{MCMC_CONFIG['burnin']}_th{MCMC_CONFIG['thin']}"
           f"_NKDE{KDE_CONFIG['N_KDE']}_seed{MCMC_CONFIG['seed']}{model.free_tag()}")
print("RUN_TAG:", RUN_TAG)

## 3. KDE self-consistency check

In [ ]:
import matplotlib.pyplot as plt
plot.plot_kde_ppc(model, planet=PLANET, paths=paths, run_tag=RUN_TAG); plt.show()

## 4. Run the MCMC

In [ ]:
try:    import multiprocess as mp
except ImportError: import multiprocessing as mp
try:    _ctx = mp.get_context("fork")
except (AttributeError, ValueError): _ctx = mp

result = pr.run_emcee(model, MCMC_CONFIG, ctx=_ctx)
print(f"\nacc frac = {result['acc_frac']*100:.1f}%  | runtime {result['runtime_s']:.1f}s | N={len(result['chain'])}")

## 5. Posterior summary

In [ ]:
for name in model.PARAM_NAMES:
    s = result["stats"][name]; m = s["median"]
    print(f"  {name:>10}: {m:.5f}  [-{m-s['p16']:.5f}, +{s['p84']-m:.5f}]")

## 6. Posterior predictive check

In [ ]:
import matplotlib.pyplot as plt
ppc = pr.compute_ppc(model, result["chain"])
run = pr.make_run(model, result, RUN_TAG, PLANET, ppc=ppc)
plot.plot_ppc(run, data["ttv"], paths=paths); plt.show()

## 7. emcee native diagnostics (log-prob trace + autocorrelation)

In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
_med = np.median(result["logprob_raw"], axis=1)
axes[0].plot(_med, lw=0.9); axes[0].axvline(MCMC_CONFIG["burnin"], color="k", ls="--", label="burn-in")
axes[0].set_xlabel("step"); axes[0].set_ylabel("median log-prob"); axes[0].legend(fontsize=8)
try:
    import emcee
    tau = emcee.autocorr.integrated_time(result["chain_raw"], tol=0)
    axes[1].bar(model.PARAM_NAMES, tau, color=plot.planet_color(PLANET), alpha=0.8)
    axes[1].set_ylabel(r"$\hat{\tau}$ (integrated autocorr.)")
    print("autocorr times:", dict(zip(model.PARAM_NAMES, np.round(tau, 1))))
except Exception as e:
    axes[1].set_axis_off(); print("autocorr failed:", e)
fig.tight_layout()
fig.savefig(paths.figures_dir("diagnostics") / f"{RUN_TAG}_diagnostics.png", dpi=plot.STYLE["fig_dpi"]); plt.show()

## 8. Marginals and corner (publication style)

In [ ]:
import matplotlib.pyplot as plt
plot.plot_marginals(run, berger_rho=data["rho_true_gcc_samples"], paths=paths); plt.show()
plot.plot_corner(run, paths=paths); plt.show()

## 9. Save results

In [ ]:
meta = dict(
    planet=PLANET, case=CASE, run_tag=RUN_TAG, sampler="emcee",
    kde_observables=model.observables, N_KDE=int(len(model.idx_train)),
    seed_kde=int(KDE_CONFIG["seed_kde"]),
    nwalkers=int(MCMC_CONFIG["nwalkers"]), nsteps=int(MCMC_CONFIG["nsteps"]),
    burnin=int(MCMC_CONFIG["burnin"]), thin=int(MCMC_CONFIG["thin"]),
    seed_mcmc=int(MCMC_CONFIG["seed"]), FORWARD_MODEL=FORWARD_MODEL,
    B_FREE=bool(model.B_FREE), B_FIXED=float(model.B_FIXED), B_SIGMA=float(model.B_SIGMA),
    RHO_TRUE_FREE=bool(model.RHO_TRUE_FREE), RHO_TRUE_FIXED=float(model.RHO_TRUE_FIXED),
    TAU_FREE=bool(model.TAU_FREE), TAU_FIXED=float(model.TAU_FIXED),
    P_FREE=bool(model.P_FREE), FI_FIXED=float(model.FI_FIXED), FE_MAX=float(model.FE_MAX),
    p_min=float(model.p_min), p_max=float(model.p_max), p_mean_ref=float(model.p_mean_ref),
    P_fixed_days=float(model.P_fixed),
    acc_frac=float(result["acc_frac"]), runtime_s=float(result["runtime_s"]),
    n_samples=int(len(result["chain"])), param_names=model.PARAM_NAMES,
)
for _n, _s in result["stats"].items():
    meta[f"stat_{_n}_median"] = float(_s["median"]); meta[f"stat_{_n}_p16"] = float(_s["p16"]); meta[f"stat_{_n}_p84"] = float(_s["p84"])

arrays = dict(chain=result["chain"], logprob=result["logprob"],
              chain_raw=result["chain_raw"], logprob_raw=result["logprob_raw"], ppc=ppc)
pr.save_run(paths.results_dir(FORWARD_MODEL), RUN_TAG, arrays, meta)
print("Saved ->", paths.results_dir(FORWARD_MODEL))